# Week 92, Day 2: Write-up

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w092d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('writeup.md', 'w', encoding='utf-8') as f:
    f.write("# Waymark: honest walking times\n\n## Problem\n\nWalkers choose routes by distance, then find that the climb doubles the time. Waymark plans routes on a map of junctions and paths. It estimates times with Naismith's rule: 12 minutes per kilometre plus 1 minute for every 10 metres of climbing.\n\n## Approach\n\nThe map is a graph. Shortest routes come from Dijkstra's algorithm, e.g. Mill to Ridge in 6.0 km. Round walks start and end at the same junction and aim for a target time. The planner searches for turning points rather than trying every loop.\n\n## Results\n\nWith A*, planning became much faster on large maps. Planning Mill to Ridge takes 4.1 ms at the median and 4.4 ms at p95, inside its 10 ms budget. Five testers finished the planning task without help. The round-walk task went from 60% to 100% success after the target time box moved to the top of the form. The mean SUS score was 72.5.\n\n## Decisions\n\nRoutes are stored as lists of junction names rather than coordinates, because the map can change and a saved route should still open, and because names are what walkers read and type, which also made the share codes, the GeoJSON export and the walk log simpler to build and to test across all three sprints.\n\n## What next\n\nExport to GPX was cut in the first sprint. It comes next, then maps from open data.\n")

**Predict before you run:** what will this print?



In [ ]:
text = "The loop is 2.5 km. It takes 40 minutes."
print(len([s for s in text.split(".") if s.strip()]))

> **Key idea.** Structure a write-up as problem, approach, decisions, results, limitations, what next, in 800 to 1500 words. Measure sentences with a careful splitter (abbreviations, numbers) and keep the average under about 20 words, splitting any over 25. Back every comparison with a number and how it was measured. Record each important decision as an ADR: context, decision, consequences.

### Exercise: The outline

Write `outline_problems(markdown, required=REQUIRED)`: the sections are the lines starting with `## ` (the text after it, compared ignoring case). Return `"missing: NAME"` for each required section that's absent, in the required order, then, for each pair of **consecutive** present required sections (in the required order) that appear the wrong way round, `"out of order: A should come before B"`.

In [ ]:
REQUIRED = ["Problem", "Approach", "Decisions", "Results", "Limitations", "What next"]

def outline_problems(markdown, required=REQUIRED):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_draft():\n    \"Waymark's first draft\"\n    assert outline_problems(open(\"writeup.md\").read()) == [\"missing: Limitations\", \"out of order: Decisions should come before Results\"]\n\ndef test_good():\n    \"everything in order\"\n    text = \"# T\\n\\n## Problem\\n\\n## approach\\n\\n## Decisions\\n\\n## Results\\n\\n## Limitations\\n\\n## What next\\n\"\n    assert outline_problems(text) == []\n    assert outline_problems(\"## B\\n## A\\n\", [\"A\", \"B\", \"C\"]) == [\"missing: C\", \"out of order: A should come before B\"]")

### Exercise: A careful sentence splitter

Write `sentences(text)`: first collapse all whitespace to single spaces; then split after `.`, `!` or `?` followed by whitespace. Join a piece back onto the previous one if the previous one's last word (ignoring case) is `e.g.`, `i.e.`, `dr.`, `mr.` or `mrs.`, or if it's `etc.`, `vs.` or `no.` and the piece starts with a lower-case letter or a digit. Return the list (empty for empty text).

In [ ]:
import re

def sentences(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_split():\n    \"abbreviations and numbers\"\n    assert sentences(\"The loop is 2.5 km. It takes 40 minutes.\") == [\"The loop is 2.5 km.\", \"It takes 40 minutes.\"]\n    assert sentences(\"Waymark plans walks, e.g. Mill to Ridge. It works!\") == [\"Waymark plans walks, e.g. Mill to Ridge.\", \"It works!\"]\n    assert sentences(\"We tested maps, paths etc. and then routes.\") == [\"We tested maps, paths etc. and then routes.\"]\n    assert sentences(\"We tested maps, paths etc. Then routes.\") == [\"We tested maps, paths etc.\", \"Then routes.\"]\n    assert sentences(\"See No. 4 on the map.  Really?\\nYes.\") == [\"See No. 4 on the map.\", \"Really?\", \"Yes.\"]\n    assert sentences(\"\") == [] and sentences(\"   \") == []")

### Exercise: Measuring sentences

Using the given `sentences` and `WORD` pattern (a word is letters and digits, perhaps joined by `'`, `.` or `-`, so `2.5` and `round-walk` are one word each), write `readability(text)` returning a dict: `sentences` (the count), `words` (the total), `per_sentence` (words per sentence to 1 decimal place, or `0.0`), and `long`: for each sentence of more than 25 words, its first five space-separated words followed by `" ..."`.

In [ ]:
import re

ALWAYS = {"e.g.", "i.e.", "dr.", "mr.", "mrs."}
SOMETIMES = {"etc.", "vs.", "no."}
WORD = re.compile(r"[A-Za-z0-9]+(?:['’.-][A-Za-z0-9]+)*")

def sentences(text):
    text = " ".join(text.split())
    if not text:
        return []
    out = []
    for piece in re.split(r"(?<=[.!?])\s+", text):
        if out:
            last = out[-1].split()[-1].lower()
            if last in ALWAYS or (last in SOMETIMES and (piece[0].islower() or piece[0].isdigit())):
                out[-1] += " " + piece
                continue
        out.append(piece)
    return out

def readability(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_draft():\n    \"Waymark's draft\"\n    text = \"\\n\".join(line for line in open(\"writeup.md\").read().splitlines() if not line.startswith(\"#\"))\n    assert readability(text) == {\"sentences\": 15, \"words\": 222, \"per_sentence\": 14.8, \"long\": [\"Routes are stored as lists ...\"]}\n\ndef test_small():\n    \"edge cases\"\n    assert readability(\"\") == {\"sentences\": 0, \"words\": 0, \"per_sentence\": 0.0, \"long\": []}\n    assert readability(\"The round-walk is 2.5 km. Don't rush.\")[\"words\"] == 7")

### Exercise: Claims without numbers

Write `unsupported_claims(text, split=None)`: return the sentences (from `split(text)`, or the given `sentences` if `split` is `None`) that contain one of the words `faster`, `slower`, `better`, `worse`, `more`, `fewer`, `less`, `improved` or `reduced` (as a whole word, ignoring case) but no digit.

In [ ]:
import re

def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", " ".join(text.split())) if s]

def unsupported_claims(text, split=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_claims():\n    \"comparisons need numbers\"\n    text = (\"With A*, planning became much faster on large maps. A* cut planning from 61 ms to 44 ms. \"\n            \"The new form is better. Testers made fewer mistakes: 2 instead of 7. Moreover, it looks fine.\")\n    assert unsupported_claims(text) == [\"With A*, planning became much faster on large maps.\", \"The new form is better.\"]\n    assert unsupported_claims(\"More! Less.\", split=lambda t: [\"More!\", \"Less.\"]) == [\"More!\", \"Less.\"]")

### Exercise: A decision record

Write `adr(number, title, status, context, decision, consequences)`, returning Markdown: `# NUMBER. TITLE`, a blank line, `Status: STATUS`, then `## Context`, `## Decision` and `## Consequences` sections, each heading followed by a blank line and its text (the consequences as `- ` bullets, one per item), with a blank line between sections.

In [ ]:
def adr(number, title, status, context, decision, consequences):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_adr():\n    \"Nygard's format\"\n    text = adr(3, \"Store routes as junction names\", \"Accepted\",\n               \"Maps change, and saved routes must still open.\", \"Store each route as a list of junction names.\",\n               [\"Routes survive map edits.\", \"Opening a route needs the map.\"])\n    assert text == (\"# 3. Store routes as junction names\\n\\nStatus: Accepted\\n\\n## Context\\n\\nMaps change, and saved routes must still open.\\n\\n\"\n                    \"## Decision\\n\\nStore each route as a list of junction names.\\n\\n## Consequences\\n\\n- Routes survive map edits.\\n- Opening a route needs the map.\")")

### Bug hunt: Three sentences in two

Splitting on every full stop cuts `2.5` in half, so this counts `3` sentences. Split only where a full stop is followed by whitespace, so it prints `2`.

In [ ]:
import re

text = "The loop is 2.5 km. It takes 40 minutes."
print(len([s for s in text.split(".") if s.strip()]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"numbers aren't sentence ends\"\n    expect_output(\"2\")")

### Bug hunt: An average rounded away

25 words in 2 sentences is 12.5 words per sentence, but this prints `12`. Fix the division so it prints `12.5`.

In [ ]:
words, count = 25, 2
print(words // count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"true division\"\n    expect_output(\"12.5\")")

### Puzzle: Writing a decision record

Arrange `adr` so the record has its title, status and three sections. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # len(sentences("Mill, e.g. Weir. Then Ridge.")) with today's splitter
q = ___   # the write-up section that says plainly what doesn't work yet
r = ___   # the three parts of an ADR after its status, as a list
s = ___   # True or False: "Planning is faster now." is a claim with evidence
t = ___   # the largest number of words a sentence may have before it counts as long

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == \"Limitations\" and r == [\"Context\", \"Decision\", \"Consequences\"], \"p, q and r\"\n    assert s is False and t == 25, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: reviewing the draft

Write `writeup_review(markdown, max_words=1500)` with today's helpers (given). The **body** is every line not starting with `#`. Return `(lines, verdict)`:

1. `"outline: ok"`, or `"outline: "` and the outline problems joined by `"; "`;
2. `"length: N words"` (words counted with `WORD` in the body), with `" (over MAX)"` added if N is more than `max_words`;
3. `"readability: X words per sentence, L long"` from `readability` on the body;
4. `"claims: ok"`, or `"claims: N without numbers"`.

`verdict` is `"ready to publish"` if the outline is ok, the length isn't over, X is at most 20 with no long sentences, and the claims are ok; otherwise `"revise: "` and the names of the failing checks (`outline`, `length`, `readability`, `claims`) joined by `", "`.

In [ ]:
import re

REQUIRED = ["Problem", "Approach", "Decisions", "Results", "Limitations", "What next"]
ALWAYS = {"e.g.", "i.e.", "dr.", "mr.", "mrs."}
SOMETIMES = {"etc.", "vs.", "no."}
WORD = re.compile(r"[A-Za-z0-9]+(?:['’.-][A-Za-z0-9]+)*")
CLAIMS = {"faster", "slower", "better", "worse", "more", "fewer", "less", "improved", "reduced"}

def outline_problems(markdown, required=REQUIRED):
    headings = [line[3:].strip().lower() for line in markdown.splitlines() if line.startswith("## ")]
    problems = [f"missing: {name}" for name in required if name.lower() not in headings]
    present = [name for name in required if name.lower() in headings]
    for a, b in zip(present, present[1:]):
        if headings.index(a.lower()) > headings.index(b.lower()):
            problems.append(f"out of order: {a} should come before {b}")
    return problems

def sentences(text):
    text = " ".join(text.split())
    if not text:
        return []
    out = []
    for piece in re.split(r"(?<=[.!?])\s+", text):
        if out:
            last = out[-1].split()[-1].lower()
            if last in ALWAYS or (last in SOMETIMES and (piece[0].islower() or piece[0].isdigit())):
                out[-1] += " " + piece
                continue
        out.append(piece)
    return out

def readability(text):
    found = sentences(text)
    counts = [len(WORD.findall(s)) for s in found]
    words = sum(counts)
    return {"sentences": len(found), "words": words, "per_sentence": round(words / len(found), 1) if found else 0.0,
            "long": [" ".join(s.split()[:5]) + " ..." for s, n in zip(found, counts) if n > 25]}

def unsupported_claims(text):
    return [s for s in sentences(text)
            if {w.lower() for w in re.findall(r"[A-Za-z]+", s)} & CLAIMS and not re.search(r"\d", s)]

def writeup_review(markdown, max_words=1500):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_draft():\n    \"Waymark's first draft\"\n    lines, verdict = writeup_review(open(\"writeup.md\").read())\n    assert lines == [\"outline: missing: Limitations; out of order: Decisions should come before Results\", \"length: 222 words\",\n                     \"readability: 14.8 words per sentence, 1 long\", \"claims: 1 without numbers\"]\n    assert verdict == \"revise: outline, readability, claims\"\n\ndef test_ready():\n    \"a short, clean write-up\"\n    sections = [\"Problem\", \"Approach\", \"Decisions\", \"Results\", \"Limitations\", \"What next\"]\n    text = \"# Waymark\\n\\n\" + \"\\n\\n\".join(f\"## {s}\\n\\nA* cut planning from 61 ms to 44 ms.\" for s in sections)\n    assert writeup_review(text) == ([\"outline: ok\", \"length: 54 words\", \"readability: 9.0 words per sentence, 0 long\", \"claims: ok\"], \"ready to publish\")\n    assert writeup_review(text, max_words=50)[0][1] == \"length: 54 words (over 50)\"\n    assert writeup_review(text, max_words=50)[1] == \"revise: length\"")

## Warm-up

Splitting text into sentences looks like splitting on full stops until a number, an abbreviation such as "e.g." or a question mark turns up. Today's tools measure writing, so they have to get that right first.

## What a write-up is for

A good capstone write-up answers, in order:

1. **Problem**: who has it, and why it matters, in a paragraph.
2. **Approach**: how Waymark solves it, at the level of ideas (a graph, Dijkstra, Naismith's rule), with one diagram.
3. **Decisions**: the three or four choices that shaped it, each with the alternatives and why you chose as you did.
4. **Results**: what it does now, with numbers: budgets met, test counts, what users managed.
5. **Limitations**: what it doesn't do and what's still wrong. Readers trust a write-up more when it says this plainly.
6. **What next**.

Keep it short: 800 to 1500 words. Link to the repository, the release and the demo rather than repeating them.

## Sentences people can follow

Readers lose the thread in long sentences. Aim for an **average** below about 20 words, and split any sentence longer than 25. Measuring needs a careful sentence splitter: split after `.`, `!` or `?` followed by a space, but don't split after abbreviations that never end a sentence (`e.g.`, `i.e.`, `Dr.`), and treat `etc.`, `vs.` and `No.` as ending a sentence only when a capital letter follows. Numbers such as `2.5` contain no space after the point, so they're safe.

Prefer plain words ("use", not "utilise"), active verbs ("Waymark stores routes", not "routes are stored by Waymark"), and one idea per sentence.

## Claims and evidence

"A* made planning much faster" invites the question: how much? Every comparison (faster, better, fewer, improved) should come with a **number** and how it was measured: "A* cut planning on a 30 by 30 map from 61 ms to 44 ms at p95". The numbers already exist: the budgets, the benchmarks, the user tests. A program can't judge a claim, but it can find comparisons in sentences with no digits in them, which is a good list to reread.

## Decision records

An **architecture decision record** (ADR, from Michael Nygard) is a short file per decision: a numbered title, a status (proposed, accepted, superseded), the **context** (what forced a choice), the **decision**, and the **consequences**, good and bad. Kept in `docs/decisions/`, they let the write-up's Decisions section link to the full story, and they explain to the next person why the code is the way it is.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Draft your write-up** in the six sections, 800 to 1500 words, with one diagram.
2. **Write three ADRs** for the decisions that shaped your project, and link them from the Decisions section.
3. **Review it** with `writeup_review`, then ask someone who doesn't program to read the Problem and Results sections and tell you what they understood.
4. **Revise** until every claim has a number, then publish it in the repository's `docs/` and link it from the README.

## Before you go

Tomorrow you'll show Waymark working, in a short demo video with a script, captions and chapters.